# Results template

Load artifacts from a Hydra run directory and reproduce the standard plots.
Point `RUN_DIR` at any folder under `outputs/`; the default picks the latest run.

In [ ]:
import sys
from pathlib import Path

import numpy as np
from omegaconf import OmegaConf

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'Notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from isbm.metrics.clustering import ari
from isbm.utils.plotting import ll_trace_plot, sorted_adjacency_plot

In [ ]:
outputs_root = PROJECT_ROOT / 'outputs'
candidates = sorted(outputs_root.glob('*/*'), key=lambda p: p.stat().st_mtime)
RUN_DIR = candidates[-1]
print('Loading run:', RUN_DIR.relative_to(PROJECT_ROOT))

In [ ]:
cfg = OmegaConf.load(RUN_DIR / '.hydra' / 'config.yaml')
artifacts = np.load(RUN_DIR / 'artifacts.npz')
X = artifacts['X']
Z_true = artifacts['Z_true']
z1 = artifacts['z1']
z2 = artifacts['z2']
ll_trace = artifacts['ll_trace']

print(OmegaConf.to_yaml(cfg))
print(f'ARI(true, z1) = {ari(Z_true, z1):.3f}')
print(f'final pseudo-LL = {ll_trace[-1]:.2f}')

In [ ]:
ll_trace_plot(ll_trace, burnin=cfg.model.burnin);

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
sorted_adjacency_plot(X, Z_true, ax=axes[0], title='Sorted by ground-truth Z')
sorted_adjacency_plot(X, z1, ax=axes[1], title='Sorted by inferred z1')
plt.tight_layout();